# Schizophrenia Pathway Classifier — Classification Modeling

## Objective

In this notebook I am constructing a classification model trained on the expression data to test if the model learns features reflecting the enrichment of immune/inflammatory pathways in differentially expressed genes between SCZ and control within the dataset found in the previous notebook — directly answering the second part of my primary hypothesis. I will engineer features, make preprocessing decisons, and use cross validation to evaluate the models performance. High overlap between the classifier's top features and the Hallmark leading-edge genes from `02_enrichment_analysis.ipynb`'s GSEA (`Lead_genes`) will be evidence in support of the second part of the primary hypothesis, if low or no overlap is found then this test failed to find supporting evidence for this hypothesis.

## Inputs
- `../data/processed/merged_df.csv`
- `../data/processed/gsea_results.csv`
- `../data/processed/hallmark_gene_sets.json`
- `../data/processed/probe_to_gene.json`

## Output
- Features list with per-fold selected gene sets
- Evaluation table with per-sample LOO predictions and metrics
- Feature importance table

## 3.1 Setup & Load Data
Import many of the same libraries from the previous notebooks, with addition to `sklearn` imports for modeling, preprocessing, and evaluating. I set paths and random state for reproducability, and load them into the notebook. 

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from scipy import stats

from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score

RANDOM_STATE = 42
PROCESSED_DATA_PATH = '../data/processed/merged_df.csv'
GSEA_RESULTS_PATH = '../data/processed/gsea_results.csv'
HALLMARK_PATH = '../data/processed/hallmark_gene_sets.json'
PROBE_TO_GENE_PATH = '../data/processed/probe_to_gene.json'

In [18]:
merged_df = pd.read_csv(PROCESSED_DATA_PATH)
# check shape and value counts to confirm df was imported correctly
print(merged_df.shape)
print(merged_df['diagnosis'].value_counts())

gsea_results = pd.read_csv(GSEA_RESULTS_PATH)
# check shape and columns to confirm df was imported correctly
print(gsea_results.shape)
print(gsea_results.columns)

hallmark_set = json.load(open(HALLMARK_PATH))
# check length and keys to confirm json was imported correctly
print(len(hallmark_set))
print(list(hallmark_set.keys())[:10])

probe_to_gene = json.load(open(PROBE_TO_GENE_PATH))
# check length and keys to confirm json was imported correctly
print(len(probe_to_gene))
print(list(probe_to_gene.keys())[:10])

(59, 30065)
diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64
(5, 10)
Index(['Name', 'Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'FWER p-val',
       'Tag %', 'Gene %', 'Lead_genes'],
      dtype='str')
5
['HALLMARK_INFLAMMATORY_RESPONSE', 'HALLMARK_INTERFERON_ALPHA_RESPONSE', 'HALLMARK_INTERFERON_GAMMA_RESPONSE', 'HALLMARK_IL6_JAK_STAT3_SIGNALING', 'HALLMARK_COMPLEMENT']
54675
['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at', '1294_at', '1316_at', '1320_at', '1405_i_at', '1431_at']


`merged_df` returns the correct shape (59, 30065), and has the correct diagnosis distribution, confirming it was loaded correctly. `gsea_results` has the correct number of rows (5 for the five Hallmark sets) and the correct column count (10) and names. `hallmark_set` has returns the correct 5 keys. `probe_to_gene` has the correct length of 54675, matching it's length in `02_enrichment_analysis.ipynb`.

## 3.2 Feature Engineering



In [ ]:
# probes to mean expression : chose mean since the small smaple size would make a variance approach unreliable
probe_means = merged_df.drop(columns=['sample_id', 'diagnosis', 'duration', 'name']).mean()
gene_symbol = probe_means.index.map(probe_to_gene)
data = {'gene symbol': gene_symbol, 'probe mean': probe_means}
df =  pd.DataFrame(data)

In [20]:
df.shape

(30061, 2)

In [21]:
df['gene symbol'].isna().sum()

np.int64(4899)

In [23]:
df['gene symbol'].value_counts().head(10)

gene symbol
MALAT1    12
ZBTB20    11
QKI       10
FGFR2     10
MSI2       9
MEG3       9
PPARA      9
DNAH1      9
HCG18      9
STRN       9
Name: count, dtype: int64

In [24]:
df[df['gene symbol'] == 'MALAT1']

,gene symbol,probe mean
1558678_s_at,MALAT1,12.173298
223577_x_at,MALAT1,8.695314
223578_x_at,MALAT1,7.568805
223940_x_at,MALAT1,9.859788
224558_s_at,MALAT1,9.165053
224559_at,MALAT1,8.886853
224567_x_at,MALAT1,10.395358
224568_x_at,MALAT1,9.341297
226675_s_at,MALAT1,8.959744
227510_x_at,MALAT1,9.042417


Verifies theres meanigful difference in probe mean expresion values, confirming they are henuinely distinct measurments not noise

In [12]:
probe_means.index[:5]

Index(['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at'], dtype='str')

In [13]:
list(probe_to_gene.keys())[:5]

['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at']

In [ ]:
probe_means.head() 

1007_s_at    9.462686
1053_at      6.798064
117_at       4.879919
121_at       7.763376
1255_g_at    5.211624
dtype: float64

In [15]:
idx_val = probe_means.index[0]
dict_key = list(probe_to_gene.keys())[0]

print(type(idx_val), type(dict_key))
print(idx_val == dict_key)
print(idx_val in probe_to_gene)

<class 'str'> <class 'str'>
True
True


In [14]:
probe_to_gene['224264_x_at']

'ZAN'

## 3.3 Preprocessing Decisions

## 3.4 Modeling + Cross-Validation — Leave-one-out (LOO) 

## 3.5 Evaluation

## 3.6 Feature Importance

## 3.7 Summary